# Phase 4 — Native resolution 2048 (R50, then X101)

Until now every model saw the images shrunk 2048 → 1024, so small filaments were lost. Here both models train and
predict on the **full 2048×2048 image** (no tiles). Everything else is the `r50_allann` recipe (all annotators'
labels, 6,000 iterations, batch 2, same anchors), so the score differences come from resolution and backbone only.

| Step | Name | What | Time on A100 (estimate) |
|---|---|---|---|
| 0 | `probe_x101_2048` | sanity check: 20 X101 training steps + predictions on 7 images; prints GPU memory and speed | ~10 min |
| 1 | `r50_2048` | R50-FPN at 2048 → raw val + test predictions | ~1.5–2 h |
| 2 | `x101_2048` | X101-FPN at 2048 → raw val + test predictions | ~3 h |

**Runtime:** A100 GPU + High-RAM (Runtime → Change runtime type). ~8 compute units per hour, ~40–45 units in total.
Close any other Colab sessions first. Keep this browser tab open while it runs.

**How to run:** Runtime → Run all. Every step stops the notebook if a command fails, so nothing runs on a broken
setup. Raw predictions go to `MyDrive/solar_filament/Phase 4/`; scoring is done on the local computer.

**If Colab disconnects:** run sections 1–4 again, then re-run the cell of the step that was interrupted. Training
resumes from the last checkpoint on Drive (every 1,000 iterations).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Mount Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/solar_filament"
OUT_DIR = f"{DRIVE_DIR}/output"            # checkpoints (persistent)
PHASE4_DIR = f"{DRIVE_DIR}/Phase 4"         # raw predictions to send back
os.makedirs(PHASE4_DIR, exist_ok=True)
os.environ["SOLAR_OUTPUT_DIR"] = OUT_DIR
print("Drive ready:", OUT_DIR)

## 2. Get the latest code (pulls if the repo is already cloned)

In [ ]:
BRANCH = "Dev"
REPO = "github.com/amiruliman592/Kaggle-Competition-.git"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"

if os.path.exists("/content/solar"):
    !git -C /content/solar pull -q origin {BRANCH}
else:
    !git clone -q --depth 1 -b {BRANCH} {url} /content/solar
%cd /content/solar
os.environ["SOLAR_ROOT"] = "/content/solar"
!git log --oneline -1

## 3. Install dependencies (Detectron2 build ~5-8 min)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'
import torch, detectron2
print("torch", torch.__version__, "| detectron2", detectron2.__version__, "| GPU", torch.cuda.get_device_name(0))

## 4. Quick checks (~30 s) and helpers

In [ ]:
import shutil, time

def run(args):
    """Run one run_pipeline.py command; stop the notebook if it fails."""
    t0 = time.time()
    get_ipython().system(f"python run_pipeline.py {args}")
    code = get_ipython().user_ns.get("_exit_code", 0)
    if code != 0:
        raise RuntimeError(f"Command failed (exit {code}): {args}")
    print(f"--- done in {(time.time() - t0) / 60:.1f} min")

def save_raw(name):
    for split in ("val", "test"):
        src = f"{OUT_DIR}/raw_preds_{name}_{split}.json.gz"
        shutil.copy(src, PHASE4_DIR)
        print(split, os.path.getsize(src) // 1024, "KB ->", PHASE4_DIR)

!python -m pytest -q -p no:warnings tests/test_pipeline_flags.py tests/test_official_metric.py
assert _exit_code == 0, "Tests failed - stop and tell Claude"

## Step 0 — Sanity check (`probe_x101_2048`, ~10 min)
X101 is the heavier model: if it fits and runs, R50 will too. Look for these lines in the output:
- `[Train] Peak GPU memory: ... GB` — must be well below 80 GB
- `time: ...` in the iteration log line — seconds per training step (6,000 steps × this = training time)
- `[Dump] val: ... s/img` — prediction speed at 2048

A warning that some RPN head weights were skipped is expected (as in Phase 2/3). The probe files are deleted at
the end.

In [ ]:
PROBE = "probe_x101_2048"
FLAGS_X101 = "--backbone x101 --resolution 2048"
run(f"--step train --name {PROBE} {FLAGS_X101} --label-policy all --max-iter 20 --no-resume")
run(f"--step dump --dry-run --name {PROBE} {FLAGS_X101}")
shutil.rmtree(f"{OUT_DIR}/detectron2/{PROBE}", ignore_errors=True)
for split in ("val", "test"):
    f = f"{OUT_DIR}/raw_preds_{PROBE}_{split}.json.gz"
    if os.path.exists(f):
        os.remove(f)
print("Sanity check passed.")

## Step 1 — R50-FPN at 2048 (`r50_2048`, ~1.5–2 h)
Same as `r50_allann` (val PQ 0.3979), only the resolution changes.

In [ ]:
NAME_R50 = "r50_2048"
FLAGS_R50 = "--backbone r50 --resolution 2048"
run(f"--step train --name {NAME_R50} {FLAGS_R50} --label-policy all")
run(f"--step dump  --name {NAME_R50} {FLAGS_R50}")
save_raw(NAME_R50)

## Step 2 — X101-FPN at 2048 (`x101_2048`, ~3 h)
Same settings as step 1, only the backbone changes.

In [ ]:
NAME_X101 = "x101_2048"
FLAGS_X101 = "--backbone x101 --resolution 2048"
run(f"--step train --name {NAME_X101} {FLAGS_X101} --label-policy all")
run(f"--step dump  --name {NAME_X101} {FLAGS_X101}")
save_raw(NAME_X101)

## 5. Send results back
Download the raw prediction files (also in `MyDrive/solar_filament/Phase 4/`) into the local folder
`Result Phase 4/`, then tell Claude. Please also copy the `Peak GPU memory`, the `time:` from the last training
log line of each run, and the `[Dump] ... s/img` lines. Loss curves are in
`MyDrive/solar_filament/output/detectron2/<name>/metrics.json` if needed.

In [ ]:
from google.colab import files
for name in ("r50_2048", "x101_2048"):
    for split in ("val", "test"):
        f = f"{PHASE4_DIR}/raw_preds_{name}_{split}.json.gz"
        if os.path.exists(f):
            files.download(f)
        else:
            print("not found (run not finished?):", f)